# Set up

## Change working directory

In [1]:
%cd ../

C:\Users\Skylar\Downloads\tools


C:\Users\Skylar\Downloads\tools\env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Install packages

In [2]:
# Placeholder

## Import development libraries

In [3]:
import ipytest
from sklearn import base as snbe
from sklearn import compose as snce
from sklearn import model_selection as snmos
from sklearn import pipeline as snpe
from tools import utils as tsus

ipytest.autoconfig(rewrite_asserts=False)

## Import other libraries

In [4]:
import pandas as pd
from sklearn.feature_extraction import text as snfett

## Declare constants

In [5]:
module: str = "feature_extraction"

# Define CountVectorizer

## Define

In [6]:
class CountVectorizer(snfett.CountVectorizer):
    def fit(self, X: pd.Series, y: pd.Series | None = None) -> "CountVectorizer":
        super().fit(raw_documents=X)
        return self

    def transform(self, X: pd.Series) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def fit_transform(self, X: pd.Series, y: pd.Series | None = None) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().fit_transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def set_output(self, *, transform: str | None = None) -> None:
        pass

## Demonstrate

In [7]:
def get_text_length(data: pd.DataFrame) -> pd.DataFrame:
    """This is just to show how this class functions when used as part of pipeline on dataset with additional features"""
    return data.assign(**{"text_len": lambda x: x["text"].str.len()})


def get_pipeline(transformer: snbe.BaseEstimator) -> snpe.Pipeline:
    return snpe.make_pipeline(
        snpe.FunctionTransformer(func=get_text_length),
        snce.make_column_transformer((transformer, "text"), remainder="passthrough"),
    ).set_output(transform="pandas")


# Read in data
url: str = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
data: pd.DataFrame = pd.read_csv(filepath_or_buffer=url, sep="\t", names=["label", "text"])
data.info()
display(data)
# Split into target vector and feature matrix
target: str = "label"
X: pd.DataFrame = data.drop(columns=target)
y: pd.Series = data[target]
# Split into train and test sets
X_train, X_test, y_train, y_test = splits = snmos.train_test_split(X, y, test_size=2e-1, shuffle=False)  # type: list[pd.DataFrame | pd.Series]
tsus.describe_structure(x=splits)
# Initialize, fit, and transform with pipeline
pipeline: snpe.Pipeline = get_pipeline(transformer=CountVectorizer(max_features=5, stop_words="english"))
display(pipeline)
X_train_transformed: pd.DataFrame = pipeline.fit_transform(X=X_train)
X_test_transformed: pd.DataFrame = pipeline.transform(X=X_test)
display(X_train_transformed, X_test_transformed)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


<class 'list'> with 4 elements
- element: 0
  (<class 'pandas.core.frame.DataFrame'>, (4457, 1))
- element: 1
  (<class 'pandas.core.frame.DataFrame'>, (1115, 1))
- element: 2
  (<class 'pandas.core.series.Series'>, (4457,))
- element: 3
  (<class 'pandas.core.series.Series'>, (1115,))


,steps,"[('functiontransformer', ...), ('columntransformer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function get...002969DD74E50>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


,countvectorizer__gt,countvectorizer__just,countvectorizer__lt,countvectorizer__ok,countvectorizer__ur,remainder__text_len
0,0,0,0,0,0,111
1,0,0,0,1,0,29
2,0,0,0,0,0,155
3,0,0,0,0,0,49
4,0,0,0,0,0,61
...,...,...,...,...,...,...
4452,0,0,0,0,0,379
4453,0,1,0,0,0,69
4454,1,0,1,0,0,26
4455,0,0,0,0,0,36


,countvectorizer__gt,countvectorizer__just,countvectorizer__lt,countvectorizer__ok,countvectorizer__ur,remainder__text_len
4457,0,0,0,0,0,116
4458,0,1,0,0,0,50
4459,0,0,0,0,0,89
4460,0,0,0,0,0,149
4461,0,0,0,0,0,237
...,...,...,...,...,...,...
5567,0,0,0,0,0,160
5568,0,0,0,0,0,36
5569,0,0,0,0,0,57
5570,0,0,0,0,0,125


## Test

In [8]:
%%ipytest

def test_fit_transform() -> None:
    X = pd.Series(data=['hello there', 'hello world'], name='a')
    cv = CountVectorizer()
    out: pd.DataFrame = cv.fit_transform(X=X)
    assert out.shape == (2, 3)
    assert out.index.equals(other=X.index)
    assert set(out.columns) == {'hello', 'there', 'world'}

def test_fit_after_transform() -> None:
    X = pd.Series(data=['hello there', 'hello world'], name='a')
    cv = CountVectorizer().fit(X=X)
    out: pd.DataFrame = cv.transform(X=X)
    assert out.shape == (2, 3)
    assert out.index.equals(other=X.index)
    assert set(out.columns) == {'hello', 'there', 'world'}

def test_empty_strings() -> None:
    X = pd.Series(data=['', ' ', 'hello'], name='a')
    cv = CountVectorizer()
    out: pd.DataFrame = cv.fit_transform(X=X)
    assert out.shape[0] == 3
    assert out.loc[:1, :].eq(other=0).all().all()
    assert out.loc[2, 'hello'] == 1

def test_determinism() -> None:
    X = pd.Series(data=["a b c", "b c d"], name='a')
    cv = CountVectorizer(vocabulary=list('abcd'))
    pd.testing.assert_frame_equal(left=cv.fit_transform(X=X), right=cv.fit_transform(X=X))

....                                                                                         [100%]
4 passed in 0.33s


# Define TfidfVectorizer

## Define

In [9]:
class TfidfVectorizer(snfett.TfidfVectorizer):
    def fit(self, X: pd.Series, y: pd.Series | None = None) -> "TfidfVectorizer":
        super().fit(raw_documents=X)
        return self

    def transform(self, X: pd.Series) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def fit_transform(self, X: pd.Series, y: pd.Series | None = None) -> pd.DataFrame:
        self.fit(X=X, y=y)
        return self.transform(X=X)

    def set_output(self, *, transform: str | None = None) -> None:
        pass

## Demonstrate

In [10]:
# Initialize, fit, and transform with pipeline
pipeline: snpe.Pipeline = get_pipeline(transformer=TfidfVectorizer(max_features=5, stop_words="english"))
display(pipeline)
X_train_transformed: pd.DataFrame = pipeline.fit_transform(X=X_train)
X_test_transformed: pd.DataFrame = pipeline.transform(X=X_test)
display(X_train_transformed, X_test_transformed)

,steps,"[('functiontransformer', ...), ('columntransformer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,func,<function get...002969DD74E50>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


,tfidfvectorizer__gt,tfidfvectorizer__just,tfidfvectorizer__lt,tfidfvectorizer__ok,tfidfvectorizer__ur,remainder__text_len
0,0.000000,0.0,0.000000,0.0,0.0,111
1,0.000000,0.0,0.000000,1.0,0.0,29
2,0.000000,0.0,0.000000,0.0,0.0,155
3,0.000000,0.0,0.000000,0.0,0.0,49
4,0.000000,0.0,0.000000,0.0,0.0,61
...,...,...,...,...,...,...
4452,0.000000,0.0,0.000000,0.0,0.0,379
4453,0.000000,1.0,0.000000,0.0,0.0,69
4454,0.707538,0.0,0.706675,0.0,0.0,26
4455,0.000000,0.0,0.000000,0.0,0.0,36


,tfidfvectorizer__gt,tfidfvectorizer__just,tfidfvectorizer__lt,tfidfvectorizer__ok,tfidfvectorizer__ur,remainder__text_len
4457,0.0,0.0,0.0,0.0,0.0,116
4458,0.0,1.0,0.0,0.0,0.0,50
4459,0.0,0.0,0.0,0.0,0.0,89
4460,0.0,0.0,0.0,0.0,0.0,149
4461,0.0,0.0,0.0,0.0,0.0,237
...,...,...,...,...,...,...
5567,0.0,0.0,0.0,0.0,0.0,160
5568,0.0,0.0,0.0,0.0,0.0,36
5569,0.0,0.0,0.0,0.0,0.0,57
5570,0.0,0.0,0.0,0.0,0.0,125


## Test

In [11]:
%%ipytest

def test_fit_transform() -> None:
    X = pd.Series(data=['hello there', 'hello world'], name='a')
    cv = TfidfVectorizer()
    out: pd.DataFrame = cv.fit_transform(X=X)
    assert out.shape == (2, 3)
    assert out.index.equals(other=X.index)
    assert set(out.columns) == {'hello', 'there', 'world'}

def test_fit_after_transform() -> None:
    X = pd.Series(data=['hello there', 'hello world'], name='a')
    cv = TfidfVectorizer().fit(X=X)
    out: pd.DataFrame = cv.transform(X=X)
    assert out.shape == (2, 3)
    assert out.index.equals(other=X.index)
    assert set(out.columns) == {'hello', 'there', 'world'}

def test_empty_strings() -> None:
    X = pd.Series(data=['', ' ', 'hello'], name='a')
    cv = TfidfVectorizer()
    out: pd.DataFrame = cv.fit_transform(X=X)
    assert out.shape[0] == 3
    assert out.loc[:1, :].eq(other=0).all().all()
    assert out.loc[2, 'hello'] == 1

def test_determinism() -> None:
    X = pd.Series(data=["a b c", "b c d"], name='a')
    cv = TfidfVectorizer(vocabulary=list('abcd'))
    pd.testing.assert_frame_equal(left=cv.fit_transform(X=X), right=cv.fit_transform(X=X))

....                                                                                         [100%]
4 passed in 0.04s


# Write module

## Write

In [12]:
module_path: str = "src/tools/%s.py" % module

In [13]:
%%file $module_path

import pandas as pd
from sklearn.feature_extraction import text as snfett

class CountVectorizer(snfett.CountVectorizer):
    def fit(self, X: pd.Series, y: pd.Series | None = None) -> "CountVectorizer":
        super().fit(raw_documents=X)
        return self

    def transform(self, X: pd.Series) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def fit_transform(self, X: pd.Series, y: pd.Series | None = None) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().fit_transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def set_output(self, *, transform: str | None = None) -> None:
        pass

class TfidfVectorizer(snfett.TfidfVectorizer):
    def fit(self, X: pd.Series, y: pd.Series | None = None) -> "TfidfVectorizer":
        super().fit(raw_documents=X)
        return self

    def transform(self, X: pd.Series) -> pd.DataFrame:
        return pd.DataFrame(
            data=super().transform(raw_documents=X).toarray(), columns=self.get_feature_names_out(), index=X.index
        )

    def fit_transform(self, X: pd.Series, y: pd.Series | None = None) -> pd.DataFrame:
        self.fit(X=X, y=y)
        return self.transform(X=X)

    def set_output(self, *, transform: str | None = None) -> None:
        pass

Writing src/tools/feature_extraction.py


## Format

In [14]:
!ruff format $module_path

1 file reformatted


## Check with ruff

In [15]:
!ruff check $module_path

All checks passed!


## Check with ty

In [16]:
!ty check $module_path

error[invalid-method-override]: Invalid override of method `fit`
    --> src\tools\feature_extraction.py:6:9
     |
   5 | class CountVectorizer(snfett.CountVectorizer):
   6 |     def fit(self, X: pd.Series, y: pd.Series | None = None) -> "CountVectorizer":
     |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Definition is incompatible with `sklearn.feature_extraction.text.CountVectorizer.fit`
   7 |         super().fit(raw_documents=X)
   8 |         return self
     |
    ::: env\Lib\site-packages\sklearn\feature_extraction\text.py:1312:9
     |
1310 |         return vocabulary, X
1311 |
1312 |     def fit(self, raw_documents, y=None):
     |         -------------------------------- `sklearn.feature_extraction.text.CountVectorizer.fit` defined here
1313 |         """Learn a vocabulary dictionary of all tokens in the raw documents.
     |
info: This violates the Liskov Substitution Principle
info: rule `invalid-method-override` is enabled by default